# 第 8 章 · 记忆（LangChain / LangGraph）

四步演进，正好是"记忆"这件事在 LangChain 里的搬家史：

1. `ChatMessageHistory` —— 就是一个消息列表
2. `ConversationBufferMemory` —— 把它包装成能塞进 prompt 的变量
3. `LLMChain` + memory —— 链条自动读写记忆（v0.3 老 API，已搬到 `langchain_classic`）
4. LangGraph `BaseStore` —— 记忆**跨会话**存在 store 里，还能让 agent **改写自己的指令**

kernel 选 `Python (agentic-book)`。

In [1]:
import sys
from pathlib import Path

# 【适配】定位仓库根
_root = Path.cwd().resolve()
for _d in (_root, *_root.parents):
    if (_d / "common").is_dir():
        sys.path.insert(0, str(_d))
        break
from common import make_llm

# 【适配】LangChain 1.x 把 v0.3 的 memory / chains 搬到了 langchain_classic
from langchain_classic.memory import ChatMessageHistory, ConversationBufferMemory
from langchain_classic.chains import LLMChain
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_openai import ChatOpenAI

# ============ ① ChatMessageHistory：就是一个消息列表 ============
history = ChatMessageHistory()
history.add_user_message("I'm heading to New York next week.")
history.add_ai_message("Great! It's a fantastic city.")
print("① 消息条数:", len(history.messages))
for m in history.messages:
    print(f"   [{m.type}] {m.content}")

# ============ ② ConversationBufferMemory：能被塞进 prompt 的变量 ============
memory = ConversationBufferMemory()
memory.save_context({"input": "What's the weather like?"}, {"output": "It's sunny today."})
print("\n② load_memory_variables ->", memory.load_memory_variables({}))

# ============ ③ LLMChain + memory：链条自动读写 ============
# 【适配】书里用 OpenAI(...)（completion 接口）；DeepSeek 只有 chat 接口，改用 ChatOpenAI
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful travel agent. Previous conversation:\n{history}"),
    ("human", "{question}"),
])
memory3 = ConversationBufferMemory(memory_key="history", return_messages=False)
conversation = LLMChain(llm=ChatOpenAI(temperature=0, model="deepseek-flash"), prompt=prompt, memory=memory3)
print("\n③ 第 1 问 ->", conversation.invoke({"question": "我第一次去纽约，有什么推荐？"})["text"][:120])
print("③ 第 2 问（不带历史，靠 memory）->",
      conversation.invoke({"question": "我刚才说要去哪？"})["text"][:120])

# ============ ④ LangGraph BaseStore：跨会话的"程序记忆" ============
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()
store.put(("instructions",), "agent", {"instructions": "You are a helpful travel agent."})

prompt_template = """当前指令：
{instructions}

最近的对话：
{conversation}

请改进这段指令，让它更具体、更有用。只输出新的指令本身，不要解释。"""


def update_instructions(state, store):
    """读当前指令 + 近期对话 → 让模型改写 → 存回 store。"""
    namespace = ("instructions",)
    current = store.search(namespace)[0]
    prompt = prompt_template.format(
        instructions=current.value["instructions"],
        conversation=state["messages"],
    )
    new_instructions = make_llm(temperature=0.3).invoke(prompt).content.strip()
    store.put(namespace, "agent", {"instructions": new_instructions})
    return {"instructions": new_instructions}


state = {"messages": ["用户想去纽约，喜欢靠窗座位", "我推荐了靠窗的酒店和中央公园"]}
before = store.get(("instructions",), "agent").value["instructions"]
result = update_instructions(state, store)
after = store.get(("instructions",), "agent").value["instructions"]
print("\n④ 改写前的指令:", before)
print("④ 改写后的指令:", after[:200])
print("④ 确实存回 store:", after == result["instructions"])

① 消息条数: 2
   [human] I'm heading to New York next week.
   [ai] Great! It's a fantastic city.

② load_memory_variables -> {'history': "Human: What's the weather like?\nAI: It's sunny today."}



③ 第 1 问 -> 第一次去纽约，建议抓住“经典地标 + 街区体验 + 一两个博物馆/演出”，不要贪多。纽约很大，步行和地铁会占不少时间。下面给你一份通用推荐：

## 一、第一次去必打卡
- **自由女神像 & 炮台公园**：上岛需提前订票；只想远看可坐免费


③ 第 2 问（不带历史，靠 memory）-> 你刚才说要去 **纽约**，而且是第一次去。



④ 改写前的指令: You are a helpful travel agent.
④ 改写后的指令: 你是一名专业、细致的纽约旅行规划师。请根据用户偏好提供具体可执行的建议，包括预算、出行日期、同行人、兴趣、节奏、酒店区域与窗外景观、交通方式等。用户提到“靠窗座位”时，先确认是指航班/列车座位还是酒店房间景观，避免混淆。推荐酒店时说明区域、价格区间、交通便利性和靠窗/景观可行性；推荐景点时给出路线、耗时、预约要求和备选方案。信息不足时主动提问，回答保持简洁、实用、个性化。
④ 确实存回 store: True
